# Refactor utils functions with `spark.sql()`

This notebook deploys the SQL utility functions using Python `spark.sql()` and parameterized catalog names via f-strings.

It also fixes the `MAP_ENTRIES` issue by combining it with `EXPLODE(...)` where needed.


In [0]:
# Set the target catalog here
catalog = "platform_dev"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.utils")
print(f"Schema ensured: {catalog}.utils")


In [0]:
# fn_normalize_identifier
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.utils.fn_normalize_identifier(s STRING)
RETURNS STRING
RETURN UPPER(TRIM(s))
""")

print(f"Created: {catalog}.utils.fn_normalize_identifier")


In [0]:
# fn_has_pk
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.utils.fn_has_pk(pk ARRAY<STRING>)
RETURNS BOOLEAN
RETURN pk IS NOT NULL AND SIZE(pk) > 0
""")

print(f"Created: {catalog}.utils.fn_has_pk")


In [0]:
# fn_merge_strategy
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.utils.fn_merge_strategy(pk ARRAY<STRING>)
RETURNS STRING
RETURN CASE
  WHEN pk IS NOT NULL AND SIZE(pk) > 0 THEN 'PK'
  ELSE 'HASH'
END
""")

print(f"Created: {catalog}.utils.fn_merge_strategy")


In [0]:
# fn_contract_columns
# Fix: MAP_ENTRIES returns ARRAY<STRUCT<key, value>>, so we EXPLODE it.
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.utils.fn_contract_columns(
  contract MAP<STRING, STRING>
)
RETURNS TABLE (
  col_name STRING,
  data_type STRING
)
RETURN
SELECT
  entry.key AS col_name,
  entry.value AS data_type
FROM (
  SELECT EXPLODE(MAP_ENTRIES(contract)) AS entry
)
""")

print(f"Created: {catalog}.utils.fn_contract_columns")


In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.utils.fn_ddl_columns_from_contract(
  contract MAP<STRING, STRING>
)
RETURNS STRING
RETURN array_join(
  transform(
    map_entries(contract),
    entry -> concat('`', entry.key, '` ', entry.value)
  ),
  ', '
)
""")

print(f"Created: {catalog}.utils.fn_ddl_columns_from_contract")